In [1]:
# Set Up 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import os
# change working directory
os.chdir('/Users/gerardogutierrez/Desktop/Academics/Spring_2026/plant-health-status/Notebooks')

pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display

# trial 2 data

In [11]:
# new data merging 
# environmental data - night
environmental_data_night = pd.read_excel('../Data/Trial_2_Night_Data.xlsx')

# environmental data - day
environmental_data_day = pd.read_excel('../Data/Trial_2_Day_Data.xlsx')

# combine night and day and sort by timestamp
environmental_data = pd.concat(
    [environmental_data_night, environmental_data_day],
    axis=0,
    ignore_index=True
)

environmental_data.sort_values("DateTime", inplace=True)
environmental_data.reset_index(drop=True, inplace=True)

# rename columns 
environmental_data.rename(columns={
    'Atlas pH (CH0, Ion Concentration, pH)': 'Ion Concentration',
    'Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)': 'Carbon Dioxide',
    'Atlas EC (CH0, Electrical Conductivity, μS/cm)': 'Electrical Conductivity',
    'BME280 (CH0, Temperature, °C)': 'Temperature',
    'BME280 (CH1, Humidity, %)': 'Humidity',
    'BME280 (CH5, Vapor Pressure Deficit, Pa)': 'Vapor Pressure Deficit'
}, inplace=True)

# add day column 
def days_by_4(entry):
    if entry == 0:
        return 0
    elif entry in {1,2,3,4}:
        return 4
    elif entry in {5,6,7,8}:
        return 8
    elif entry in {9,10,11,12}:
        return 12
    elif entry in {13,14,15,16}:
        return 16
    elif entry in {17,18,19,20}:
        return 20
    elif entry in {21,22,23,24}:
        return 24
    elif entry in {25,26,27,28}:
        return 28

col_to_drop = ['Vapor Pressure Deficit']

env_df = environmental_data.copy()
day0_date = env_df['DateTime'].dt.floor('D').min()
env_df['Temp Days'] = (env_df['DateTime'].dt.floor('D') - day0_date).dt.days
env_df['Day'] = env_df['Temp Days'].apply(days_by_4)
env_df.drop(columns=['Temp Days'], inplace=True)
env_df.drop(columns=col_to_drop, inplace=True)

# mean dataframe for environmental data 
mean_env_df = (env_df
               .drop(columns=['DateTime'])
               .groupby('Day').mean().reset_index()
)

mean_env_df

,Day,Ion Concentration,Carbon Dioxide,Electrical Conductivity,Temperature,Humidity
0,0.0,5.873621,0.323495,1437.553398,20.285433,83.790346
1,4.0,5.902410,361.526487,1465.969196,21.323027,72.269387
2,8.0,5.871131,322.279345,1450.175267,20.846679,74.455971
3,12.0,5.879460,816.877390,1528.802423,21.652383,70.171890
4,16.0,6.091215,622.934783,1458.166551,22.691487,59.060762
5,20.0,6.117193,541.850000,1453.440746,21.163790,72.193106
6,24.0,6.014894,362.788597,1460.541546,21.147985,75.998370
7,28.0,6.034127,543.528239,1451.859829,21.287462,78.551094


In [ ]:
import pandas as pd

# function to load lettuce weight data into dataframe
def load_lettuce_weights(file_path='../Data/Trial_2_Fresh_Weight.xlsx'):
    weights_df = pd.read_excel(file_path, header=1, usecols="K:O")
    weights_df = weights_df.drop(columns=['Date'])
    weights_df = weights_df.dropna(how='all')
    return weights_df

def load_mean_nutr_env_df():
    """Join mean environmental and nutrient data on 'Days' column."""
    return pd.merge(mean_env_df.rename(columns={'Temperature': 'Temp (env)'}), mean_nutr_df.rename(columns={'Temperature': 'Temp (nutr)'}), on='Day')

In [6]:
environmental_data.dtypes

DateTime                                                     datetime64[ns]
Atlas pH (CH0, Ion Concentration, pH)                               float64
Atlas CO2 (Carbon Dioxide Gas) (CH0, Carbon Dioxide, ppm)           float64
Atlas EC (CH0, Electrical Conductivity, μS/cm)                      float64
BME280 (CH0, Temperature, °C)                                       float64
BME280 (CH1, Humidity, %)                                           float64
BME280 (CH5, Vapor Pressure Deficit, Pa)                            float64
dtype: object

In [3]:
import pandas as pd

# function to load lettuce weight data into dataframe
def load_lettuce_weights(file_path='../Data/Lettuce_FW_EC_Tracker_v3_2_.xlsx'):
    weights_df = pd.read_excel(file_path, sheet_name='Data_Collection', header=0)
    weights_df = weights_df.drop(columns=['Date'])
    weights_df = weights_df.dropna(how='all')
    return weights_df

def load_mean_nutr_env_df():
    """Join mean environmental and nutrient data on 'Days' column."""
    return pd.merge(mean_env_df.rename(columns={'Temperature': 'Temp (env)'}), mean_nutr_df.rename(columns={'Temperature': 'Temp (nutr)'}), on='Day')


In [ ]:

model_df = (load_lettuce_weights().merge(load_mean_nutr_env_df(), on='Day')
          .drop(columns=['Median Fresh Weight (g)', 'Average Fresh Weight (g)', 'New Fresh Weight (g)'])
)

target = "Total Fresh Weight (g)"

feature_cols = [
    "Carbon Dioxide",
    "Temp (env)",
    "Humidity",
    "Pressure",
    "Electrical Conductivity",
    "Volume",
    "Temp (nutr)",
    "Volume Flow Rate"
]

for lag in [1, 2]:
        model_df[f"weight_lag_{lag}"] = (
        model_df.groupby("Plant-ID")[target].shift(lag)
    )

model_df = model_df.dropna().reset_index(drop=True)

# rename all columns to lower case, remove parentheses  and replace spaces with snake case
model_df.columns = (model_df.columns.str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace(' ', '_')
)

model_df.head(30)

,day,plant-id,total_fresh_weight_g,baseline_g,carbon_dioxide,temp_env,humidity,pressure,electrical_conductivity,volume,temp_nutr,volume_flow_rate,weight_lag_1,weight_lag_2
0,8.0,6.0,40.1,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,33.1,29.1
1,8.0,2.0,43.4,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,36.3,30.1
2,8.0,8.0,43.6,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,36.5,29.8
3,8.0,1.0,44.3,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,38.1,29.7
4,8.0,6.0,45.0,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,40.1,33.1
5,8.0,5.0,45.1,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,38.5,30.2
6,8.0,4.0,46.0,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,39.2,29.3
7,8.0,7.0,46.2,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,39.2,30.4
8,8.0,9.0,46.8,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,36.9,29.6
9,8.0,3.0,49.2,25.22,514.051020,22.520816,65.247347,99282.511633,1521.816327,11551.277959,25.308837,1.087551,42.1,31.8


In [ ]:
# save model_df as csv
model_df.drop(columns=['day', 'plant-id', 'baseline_g']).to_csv('../Data/Trial_2modeling_data.csv', index=False)

# second set of data 